## 크로스 인코더 기반 리랭킹 
- 크로스 기반 리랭킹은 BERT와 같은 인코더 기반 언어 모델을 이용해서 질문과 문서가 얼마나 잘 맞는지 관련성을 평간하는 기법이다. 

In [10]:
import os 
import hashlib
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import OpenAIEmbeddings 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore 

In [11]:
file_path = "./data/투자설명서.pdf"

def create_loader(file_path: str):
    return PyPDFLoader(file_path)


def create_text_splitter(chunk_size: int, chunk_overlap: int):
    return RecursiveCharacterTextSplitter(chunk_size = chunk_size, chunk_overlap = chunk_overlap)


def create_embedding(model: str):
    return OpenAIEmbeddings(
        model= model,
        base_url= "http://localhost:1234/v1",
        api_key= "lm-studio",
        check_embedding_ctx_length=False,)

def make_doc_id(doc):
    return hashlib.md5(doc.page_content.encode("utf-8")).hexdigest()


def init_qdrant_vector(docs, embedding, collection_name):
    
    ids = [make_doc_id(doc) for doc in docs]
    
    return QdrantVectorStore.from_documents(
        documents=docs,
        embedding=embedding,
        ids=ids,
        url="http://localhost:6333",
        collection_name=collection_name,
    )
    
loader = create_loader(file_path)
doc_spliter = create_text_splitter(300, 100)
docs = loader.load_and_split(doc_spliter)
embedding = create_embedding("bge-m3")

In [12]:
vector_store = init_qdrant_vector(docs, embedding, "cross_invest")


In [15]:
from pydantic import BaseModel, Field
from langchain_core.documents import Document
from typing import List, Dict, Any, Tuple 
from langchain_openai import ChatOpenAI
from sentence_transformers import CrossEncoder 
from langchain_core.retrievers import BaseRetriever
from langchain_classic.chains import RetrievalQA

In [17]:

crossencoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L12-v2")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9833.15it/s]


In [19]:
class Retriever_with_cross_encoder(BaseRetriever):
    vectorstore: Any = Field(description="초기 검색을 위한 벡터 저장소")
    crossencoder: Any = Field(description="재순위화를 위한 클로스 인코더 모델 ")
    k : int = Field(default=5, description="초기에 검색할 문서 수 ") 
    rerank_top_k : int = Field(default=2, description="재순위화 후 최종적으로 반환할 문서 수")
    
    class Config:
        arbitrary_types_allowed = True 
    
    def _get_relevant_documents(self, query) -> List[Document]:
        initial_docs = self.vectorstore.similarity_search(query, k=self.k)
        pairs =[[query, doc.page_content] for doc in initial_docs]
        
        scores = self.crossencoder.predict(pairs)
        
        scored_docs = sorted(zip(initial_docs, scores), key=lambda x : x[1], reverse=True)
        
        return [doc for doc, _ in scored_docs[:self.rerank_top_k]]
        
        

/var/folders/kc/qm9ykcl12cv910wgvjsx9hs40000gn/T/ipykernel_29698/2979833281.py:1: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  class Retriever_with_cross_encoder(BaseRetriever):


In [20]:
def get_model(model : str = "gemma-3-4b-it"):
    return ChatOpenAI(
        model=model,
        temperature=0.2, 
        base_url="http://localhost:1234/v1",
        api_key="lm-studio"
)

In [21]:
cross_encode_retriever = Retriever_with_cross_encoder(
    vectorstore = vector_store, 
    crossencoder = crossencoder,
    k = 4,
    rerank_top_k = 2 
)

In [23]:
llm = get_model()

qa_chain = RetrievalQA.from_chain_type(
    llm = llm,
    chain_type="stuff", 
    retriever=cross_encode_retriever, 
    return_source_documents=True 
)

In [24]:
query = "이 회사의 2022년 영업 손실이 정확히 얼마야?"
result = qa_chain({"query": query})

/var/folders/kc/qm9ykcl12cv910wgvjsx9hs40000gn/T/ipykernel_29698/4096034051.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = qa_chain({"query": query})


In [26]:
print(result["result"])

for i, doc in enumerate(result["source_documents"]):
    print(f"\nDocument {i+1}")
    print(doc.page_content)

2022년 영업손실은 **149.1억원**이 발생했습니다.

Document 1
하여 2021년 영업손실 130.1억원, 2022년 영업손실 149.1억원, 2023년 영업손실 122억원, 2024년
1분기 영업손실 24.2억원이 발생하였습니다. 또한 영업 외적 측면에서도, 금융비용 등의 발생 영향
으로 인해 2021년 당기순손실 130.7억원, 2022년 당기순손실 228.7억원, 2023년 당기순손실
116.1억원, 2024년 1분기 당기순손실 32.9억원이 발생하는 등 지속적인 적자 구조를 면하지 못하고
있습니다.따라서 당사의 파이프라인에서 임상 성공을 통한 기술이전, 상품화 성공 등의 성과를 이루

Document 2
외적 측면에서도, 금융비용 등의 발생 영향으로 인해 2021년 당기순손실 130.7억원, 2022년
당기순손실 228.7억원, 2023년 당기순손실 116.1억원, 2024년 1분기 당기순손실 32.9억원
이 발생하는 등 지속적인 적자 구조를 면하지 못하고 있습니다. 따라서 당사의 파이프라인에
서 임상 성공을 통한 기술이전, 상품화 성공 등의 성과를 이루어내지 못한다면 당사의 적자
구조를 개선하는 것은 불가능할 수 있으며, 자본력이 지속적으로 감소하여 지속적인 연구개
